# Chat feature discovery and adaptive steering showcase

This notebook is a lightweight, fully local walkthrough of the agent capabilities requested in the project brief.

## What this notebook demonstrates
- discovering a new interaction feature from a chat transcript
- using that discovered feature to monitor drift from an original goal
- adaptively steering **toward** the feature when the draft drifts away
- adaptively steering **away** from the same feature when the goal changes

To keep the notebook fast and explanatory, it uses the shared `activation_steering` data structures with a deterministic notebook-local executor instead of a live model.


## Scenario

Assume an earlier chat established a useful interaction pattern:

> when the task asks for grounding, good answers use retrieved context and respond as a numbered checklist.

We will let the memory discover that pattern from a few chat turns, then treat it as a feature we can monitor against later goals.


In [ ]:
from pprint import pprint

import torch

import activation_steering as steering

DEMO_MODEL_NAME = "notebook-demo-model"


def make_trace(prompt: str, output_text: str, controller_id: str | None = None):
    return steering.ActivationTrace(
        model_name=DEMO_MODEL_NAME,
        controller_id=controller_id,
        layer_idx=0,
        prompt=prompt,
        top_feature_scores=[],
        output_text=output_text,
    )


## Step 1: discover a feature from prior chat turns

The memory learns interaction features from prompt/output pairs. Here the seed transcript repeatedly shows a contextual question answered as a numbered list, so the discovery step should recover a feature that captures that pattern.


In [ ]:
memory = steering.InMemorySteeringMemory()

seed_traces = [
    make_trace(
        prompt=(
            "Context:
"
            "Evidence: Paris is the capital of France.
"
            "Retrieved: city overview.

"
            "Question: What answer should we give?"
        ),
        output_text=(
            "1. Paris is the answer.
"
            "2. Evidence: the retrieved note identifies Paris as France's capital."
        ),
    ),
    make_trace(
        prompt=(
            "Context:
"
            "Evidence: The Louvre is in Paris.
"
            "Retrieved: museum overview.

"
            "Question: What supporting fact should we cite?"
        ),
        output_text=(
            "1. Cite the Louvre example.
"
            "2. Tie it back to Paris as the grounded answer."
        ),
    ),
]

for trace in seed_traces:
    memory.observe_interaction(trace)

dynamic_features = memory.list_dynamic_features(DEMO_MODEL_NAME)
pprint([
    {
        "feature_id": feature.feature_id,
        "summary": feature.summary,
        "observations": feature.observation_count,
        "metadata": feature.metadata,
    }
    for feature in dynamic_features
])

target_feature = next(
    feature
    for feature in dynamic_features
    if feature.metadata["context_usage"] == "contextual"
    and feature.metadata["output_shape"] == "list_response"
)

print("
Target feature for drift monitoring:", target_feature.feature_id)


## Step 2: define the original goal and a drift monitor

The original goal is intentionally tied to the discovered feature:

- stay grounded in retrieved evidence
- answer in a numbered checklist format

The drift monitor simply checks whether the discovered feature is present when the goal requires it, or absent when the goal says to avoid it.


In [ ]:
original_goal = {
    "name": "grounded_checklist",
    "description": "Stay grounded in retrieved evidence and answer as a numbered checklist.",
    "preference": "require",
    "target_feature_id": target_feature.feature_id,
}


def monitor_drift(goal: dict[str, str], trace: steering.ActivationTrace) -> dict[str, object]:
    feature_present = goal["target_feature_id"] in trace.observed_feature_ids
    should_include = goal["preference"] == "require"
    drift_score = 0.0 if feature_present == should_include else 1.0
    return {
        "goal": goal["name"],
        "preference": goal["preference"],
        "target_feature_id": goal["target_feature_id"],
        "feature_present": feature_present,
        "observed_feature_ids": list(trace.observed_feature_ids),
        "drift_score": drift_score,
        "explanation": (
            "Trace matches the goal-aligned feature preference."
            if drift_score == 0.0
            else "Trace drifted away from the goal-aligned feature preference."
        ),
    }


pprint(original_goal)


## Step 3: build notebook-local controllers and an adaptive agent

We register two reusable controllers:

- `grounded_checklist_boost` steers **toward** the discovered feature
- `grounded_checklist_suppress` steers **away** from the discovered feature

The executor is deterministic on purpose: it makes the feature changes obvious without needing a large-model download.


In [ ]:
toward_controller = steering.SteeringController(
    controller_id="grounded_checklist_boost",
    feature_name="grounded_checklist",
    layer_idx=0,
    vector=torch.tensor([1.0, 0.0]),
    task_types=("qa",),
    metadata={"summary": "Push answers toward contextual numbered evidence."},
)

away_controller = steering.SteeringController(
    controller_id="grounded_checklist_suppress",
    feature_name="grounded_checklist",
    layer_idx=0,
    vector=torch.tensor([-1.0, 0.0]),
    task_types=("qa",),
    metadata={"summary": "Pull answers away from the grounded checklist feature."},
)

memory.register_controllers([toward_controller, away_controller])

goal_state = dict(original_goal)
goal_state["mode"] = "neutral"


class NotebookExecutor:
    model_name = DEMO_MODEL_NAME

    def execute(self, task, plan, context, controller=None, controllers=(), max_new_tokens=80):
        prompt = "Context:
" + "
".join(context) + f"

Question: {task}"
        if controller is None:
            output_text = (
                "Paris is probably the answer, but this draft is not explicitly organized around "
                "the retrieved evidence."
            )
        elif controller.controller_id == "grounded_checklist_boost":
            output_text = (
                "1. Paris is the answer.
"
                "2. Evidence: the retrieved context states that Paris is the capital of France."
            )
        elif controller.controller_id == "grounded_checklist_suppress":
            output_text = (
                "A more exploratory direction is to riff on Parisian themes and brainstorm "
                "creative angles instead of citing the retrieved notes as a checklist."
            )
        else:
            output_text = "Unexpected controller."
        return steering.ExecutorResult(
            prompt=prompt,
            output_text=output_text,
            controller_id=controller.controller_id if controller else None,
        )


def planner(task: str, memory_store: steering.InMemorySteeringMemory) -> steering.PlannerDecision:
    if goal_state["mode"] == "toward_feature":
        return steering.PlannerDecision(
            task_type="qa",
            needs_retrieval=True,
            controller_id="grounded_checklist_boost",
            reasoning_effort="medium",
            use_steering=True,
            allow_fallback=False,
        )
    if goal_state["mode"] == "away_from_feature":
        return steering.PlannerDecision(
            task_type="qa",
            needs_retrieval=True,
            controller_id="grounded_checklist_suppress",
            reasoning_effort="medium",
            use_steering=True,
            allow_fallback=False,
        )
    return steering.PlannerDecision(
        task_type="qa",
        needs_retrieval=True,
        reasoning_effort="medium",
        use_steering=False,
        allow_fallback=False,
    )


def retriever(task: str, plan: steering.PlannerDecision) -> list[str]:
    return [
        "Evidence: Paris is the capital of France.",
        "Retrieved note: grounded answers should cite the evidence when the goal asks for grounding.",
    ]


def verifier(task, draft, context, plan):
    drift = monitor_drift(goal_state, draft.activation_trace)
    return steering.VerifierResult(
        passed=drift["drift_score"] == 0.0,
        confidence=0.9 if drift["drift_score"] == 0.0 else 0.2,
        issues=[] if drift["drift_score"] == 0.0 else [drift["explanation"]],
        metadata={"drift_score": drift["drift_score"]},
    )


agent = steering.HybridMetaCognitionAgent(
    planner=planner,
    retriever=retriever,
    executor=NotebookExecutor(),
    verifier=verifier,
    memory=memory,
)


## Step 4: detect drift, then adaptively steer toward the feature

First we run a neutral draft. It uses retrieved context, but it does **not** express the discovered checklist feature, so the drift monitor flags a mismatch against the original goal.

Then we switch the planner into `toward_feature` mode. The next run requests the boosting controller and the drift score should drop to zero.


In [ ]:
goal_state.clear()
goal_state.update(original_goal)
goal_state["mode"] = "neutral"

neutral_run = agent.run("What answer should we give about the capital of France?")
neutral_drift = monitor_drift(goal_state, neutral_run.draft.activation_trace)

goal_state["mode"] = "toward_feature"
toward_run = agent.run("What answer should we give about the capital of France?")
toward_drift = monitor_drift(goal_state, toward_run.draft.activation_trace)

for label, run, drift in [
    ("neutral draft", neutral_run, neutral_drift),
    ("steer toward feature", toward_run, toward_drift),
]:
    print(f"
=== {label.upper()} ===")
    print("selected_controller_id:", run.selected_controller_id)
    print("output:")
    print(run.draft.output_text)
    print("drift summary:")
    pprint(drift)


## Step 5: change the goal and adaptively steer away from the same feature

Now we intentionally flip the objective. Instead of wanting a grounded checklist, we want a looser creative brainstorm.

That makes the previously useful feature a source of drift, so the planner requests the suppressing controller and the monitor now prefers the feature to be absent.


In [ ]:
creative_goal = {
    "name": "creative_brainstorm",
    "description": "Brainstorm playful ideas without locking into the grounded checklist feature.",
    "preference": "avoid",
    "target_feature_id": target_feature.feature_id,
}

goal_state.clear()
goal_state.update(creative_goal)
goal_state["mode"] = "away_from_feature"

away_run = agent.run("Brainstorm a playful travel-campaign angle for France without quoting the retrieved facts.")
away_drift = monitor_drift(goal_state, away_run.draft.activation_trace)

comparison_rows = [
    {
        "label": "neutral draft",
        "selected_controller_id": neutral_run.selected_controller_id,
        "drift_score": neutral_drift["drift_score"],
        "observed_feature_ids": neutral_drift["observed_feature_ids"],
    },
    {
        "label": "steer toward feature",
        "selected_controller_id": toward_run.selected_controller_id,
        "drift_score": toward_drift["drift_score"],
        "observed_feature_ids": toward_drift["observed_feature_ids"],
    },
    {
        "label": "steer away from feature",
        "selected_controller_id": away_run.selected_controller_id,
        "drift_score": away_drift["drift_score"],
        "observed_feature_ids": away_drift["observed_feature_ids"],
    },
]

print("Output when steering away from the feature:
")
print(away_run.draft.output_text)
print("
Drift summary:")
pprint(away_drift)
print("
Comparison rows:")
pprint(comparison_rows)
print("
Controller stats:")
pprint(memory.controller_stats())


## Takeaway

The key pattern is not the specific controller IDs in this notebook. The important idea is the loop:

1. learn an interaction feature from chat traces
2. tie that feature to an explicit goal
3. monitor whether later drafts match or drift from that goal
4. choose steering that moves either toward or away from the feature as the goal changes

That is the smallest explanatory example of adaptive meta-cognitive control in this repository.
